# FFT Poisson Solver in Electrostatics

This notebook solves Poisson's equation,

$$
\nabla^2 \phi(x,y) = -\frac{\rho(x,y)}{\varepsilon_0},
$$

using the **Fast Fourier Transform (FFT)** on a **periodic 2D domain**.

The point of the notebook is to show all of the ideas in one place:

1. the physics of Poisson's equation,
2. the mathematics of the Fourier transform,
3. and the numerical implementation that turns the PDE into an algebraic solve.

The notebook has two examples:

1. a periodic manufactured-solution test,
2. and a physical dipole-like charge distribution built from two Gaussian lumps.


## Why FFT Makes Poisson Easy

In real space, Poisson's equation couples every grid point to its neighbors. A finite-difference method therefore leads to a large linear system or an iterative relaxation method.

In Fourier space, the Laplacian becomes multiplication:

$$
\mathcal{F}[\nabla^2 \phi] = -k^2 \hat{\phi},
$$

where

$$
k^2 = k_x^2 + k_y^2.
$$

So Poisson's equation becomes

$$
-k^2 \hat{\phi}(k_x,k_y) = -\frac{\hat{\rho}(k_x,k_y)}{\varepsilon_0},
$$

which gives the direct solution

$$
\hat{\phi}(k_x,k_y) = \frac{\hat{\rho}(k_x,k_y)}{\varepsilon_0\,k^2}
\qquad (k \neq 0).
$$

That is the whole idea: in the Fourier basis, the Laplacian is diagonal, so the PDE becomes a simple division mode by mode.


## Important Physics Assumption: Periodic Boundaries

An FFT Poisson solver is most natural on a **periodic box**. That means the left and right edges match, and the bottom and top edges match.

This has an important consequence: the total charge in the periodic box should be zero. Integrating Poisson's equation over the whole domain gives

$$
\int \nabla^2\phi\, dV = -\frac{1}{\varepsilon_0}\int \rho\, dV.
$$

For a periodic domain, the boundary contributions cancel, so the left-hand side is zero. Therefore the periodic problem is solvable only if

$$
\int \rho\, dV = 0.
$$

Numerically, this means the **zero-frequency mode** of the charge density must vanish. In practice, we often subtract the mean charge density before solving.

Also, the potential is determined only up to an additive constant. Setting the `k=0` mode of $\hat{\phi}$ to zero is equivalent to choosing the mean potential to be zero.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
def wave_numbers(nx, ny, lx, ly):
    """Return spectral wave-number grids for a periodic box."""
    dx = lx / nx
    dy = ly / ny
    kx = 2.0 * np.pi * np.fft.fftfreq(nx, d=dx)
    ky = 2.0 * np.pi * np.fft.fftfreq(ny, d=dy)
    return np.meshgrid(kx, ky, indexing="ij")


def solve_poisson_fft_periodic(rho, lx, ly, eps0=1.0, remove_mean=True):
    """Solve ∇²φ = -ρ/ε0 on a periodic rectangle using FFTs."""
    rho = np.asarray(rho, dtype=float)
    if rho.ndim != 2:
        raise ValueError("rho must be a 2D array")

    rho_work = rho - rho.mean() if remove_mean else rho.copy()
    rho_hat = np.fft.fftn(rho_work)
    kx, ky = wave_numbers(*rho.shape, lx, ly)
    k2 = kx**2 + ky**2

    phi_hat = np.zeros_like(rho_hat, dtype=complex)
    mask = k2 > 0.0
    phi_hat[mask] = rho_hat[mask] / (eps0 * k2[mask])
    phi_hat[~mask] = 0.0

    phi = np.fft.ifftn(phi_hat).real
    return phi, rho_work, phi_hat, rho_hat


def spectral_laplacian(field, lx, ly):
    """Compute the Laplacian spectrally on a periodic grid."""
    field = np.asarray(field, dtype=float)
    field_hat = np.fft.fftn(field)
    kx, ky = wave_numbers(*field.shape, lx, ly)
    lap_hat = -(kx**2 + ky**2) * field_hat
    return np.fft.ifftn(lap_hat).real


def electric_field_from_potential(phi, lx, ly):
    """Return E = -∇φ using spectral derivatives."""
    phi = np.asarray(phi, dtype=float)
    phi_hat = np.fft.fftn(phi)
    kx, ky = wave_numbers(*phi.shape, lx, ly)
    ex = np.fft.ifftn(-1j * kx * phi_hat).real
    ey = np.fft.ifftn(-1j * ky * phi_hat).real
    return ex, ey


def relative_l2_error(numerical, exact):
    return np.linalg.norm(numerical - exact) / np.linalg.norm(exact)


## Example 1: Manufactured Periodic Solution

To verify the method, choose a periodic exact potential:

$$
\phi_{\mathrm{exact}}(x,y) = \cos\left(\frac{2\pi x}{L_x}\right)\cos\left(\frac{2\pi y}{L_y}\right).
$$

Applying the Laplacian gives

$$
\nabla^2 \phi_{\mathrm{exact}} = -\left[\left(\frac{2\pi}{L_x}\right)^2 + \left(\frac{2\pi}{L_y}\right)^2\right]\phi_{\mathrm{exact}}.
$$

Since Poisson's equation is $\nabla^2\phi = -\rho/\varepsilon_0$, the corresponding charge density is

$$
\rho(x,y) = \varepsilon_0\left[\left(\frac{2\pi}{L_x}\right)^2 + \left(\frac{2\pi}{L_y}\right)^2\right]\phi_{\mathrm{exact}}(x,y).
$$

This example is periodic and has zero mean, so it is ideal for an FFT-based benchmark.


In [ ]:
eps0 = 1.0
nx = ny = 128
lx = ly = 1.0

x = np.linspace(0.0, lx, nx, endpoint=False)
y = np.linspace(0.0, ly, ny, endpoint=False)
X, Y = np.meshgrid(x, y, indexing="ij")

phi_exact = np.cos(2.0 * np.pi * X / lx) * np.cos(2.0 * np.pi * Y / ly)
prefactor = (2.0 * np.pi / lx) ** 2 + (2.0 * np.pi / ly) ** 2
rho = eps0 * prefactor * phi_exact

phi_fft, rho_used, _, _ = solve_poisson_fft_periodic(rho, lx, ly, eps0=eps0)
residual = spectral_laplacian(phi_fft, lx, ly) + rho_used / eps0
error = phi_fft - phi_exact

print(f"Relative L2 error: {relative_l2_error(phi_fft, phi_exact):.3e}")
print(f"Max pointwise error: {np.max(np.abs(error)):.3e}")
print(f"Residual max norm: {np.max(np.abs(residual)):.3e}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
scale = np.max(np.abs(phi_exact))
norm = TwoSlopeNorm(vmin=-scale, vcenter=0.0, vmax=scale)

im0 = axes[0].imshow(phi_exact.T, origin="lower", extent=(0, lx, 0, ly), cmap="RdBu_r", norm=norm)
axes[0].set_title("Exact potential")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
fig.colorbar(im0, ax=axes[0], shrink=0.85)

im1 = axes[1].imshow(phi_fft.T, origin="lower", extent=(0, lx, 0, ly), cmap="RdBu_r", norm=norm)
axes[1].set_title("FFT solution")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
fig.colorbar(im1, ax=axes[1], shrink=0.85)

err_scale = np.max(np.abs(error))
im2 = axes[2].imshow(error.T, origin="lower", extent=(0, lx, 0, ly), cmap="RdBu_r", vmin=-err_scale, vmax=err_scale)
axes[2].set_title("Numerical error")
axes[2].set_xlabel("x")
axes[2].set_ylabel("y")
fig.colorbar(im2, ax=axes[2], shrink=0.85)

plt.show()


## Example 2: Dipole-Like Charge Density in a Periodic Box

Now build a more physical source from two Gaussian charge clouds, one positive and one negative. This gives a neutral configuration with a clear electrostatic structure.

This is a good example because it shows:

- how the FFT solver handles smooth localized sources,
- how the potential responds to positive and negative charge,
- and how the electric field lines run from the positive lump toward the negative lump.

Because the domain is periodic, the source interacts with its periodic images. That is not a bug: it is the physical meaning of solving Poisson's equation on a torus rather than in an isolated grounded box.


In [ ]:
sigma = 0.055
rho_box = (
    1.25 * np.exp(-((X - 0.30) ** 2 + (Y - 0.58) ** 2) / (2.0 * sigma**2))
    - 1.10 * np.exp(-((X - 0.72) ** 2 + (Y - 0.42) ** 2) / (2.0 * sigma**2))
)
rho_box -= rho_box.mean()

phi_box, rho_box_used, _, _ = solve_poisson_fft_periodic(rho_box, lx, ly, eps0=eps0)
ex_box, ey_box = electric_field_from_potential(phi_box, lx, ly)
residual_box = spectral_laplacian(phi_box, lx, ly) + rho_box_used / eps0

print(f"Net charge density mean after correction: {rho_box_used.mean():.3e}")
print(f"Residual max norm: {np.max(np.abs(residual_box)):.3e}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)

rho_scale = np.max(np.abs(rho_box_used))
rho_norm = TwoSlopeNorm(vmin=-rho_scale, vcenter=0.0, vmax=rho_scale)
im0 = axes[0].imshow(rho_box_used.T, origin="lower", extent=(0, lx, 0, ly), cmap="PuOr_r", norm=rho_norm)
axes[0].set_title("Charge density $\\rho(x,y)$")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
fig.colorbar(im0, ax=axes[0], shrink=0.85)

phi_scale = np.max(np.abs(phi_box))
phi_norm = TwoSlopeNorm(vmin=-phi_scale, vcenter=0.0, vmax=phi_scale)
im1 = axes[1].imshow(phi_box.T, origin="lower", extent=(0, lx, 0, ly), cmap="coolwarm", norm=phi_norm)
axes[1].contour(X, Y, phi_box, levels=16, colors="black", linewidths=0.55, alpha=0.45)
axes[1].set_title("Potential $\\phi(x,y)$")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
fig.colorbar(im1, ax=axes[1], shrink=0.85)

axes[2].imshow(phi_box.T, origin="lower", extent=(0, lx, 0, ly), cmap="coolwarm", norm=phi_norm, alpha=0.82)
axes[2].streamplot(x, y, ex_box.T, ey_box.T, color="black", density=1.15, linewidth=0.8, arrowsize=1.0)
axes[2].contour(X, Y, phi_box, levels=16, colors="white", linewidths=0.45, alpha=0.45)
axes[2].set_title("Electric field $\\mathbf{E}=-\\nabla\\phi$")
axes[2].set_xlabel("x")
axes[2].set_ylabel("y")

plt.show()


## What This Solver Does Well

The FFT Poisson solver is excellent when:

- the domain is periodic,
- the source is smooth,
- and you want a direct, non-iterative solve.

It is often much shorter and faster than relaxation-based solvers.

## What It Does Not Replace

It does **not** replace finite-difference methods for grounded boxes or arbitrary Dirichlet boundaries. Those problems are not naturally periodic, so the FFT method is not the right tool there.

So the numerical picture is:

- **finite difference / SOR** for grounded boxes and fixed boundaries,
- **FFT / spectral methods** for periodic domains.

That is exactly why this notebook complements the finite-difference Poisson notebook rather than duplicating it.
